In [1]:
import numpy as np

# ============================================================
# RC Low-Pass Filter — Calculation (No Plots)
# ============================================================
# Given:
R = 2.2e3      # 2.2 kΩ
C = 47.0e-9    # 47.0 nF

# --- Derived parameters ---
tau = R * C                          # Time constant [s]
fc = 1.0 / (2.0 * np.pi * tau)      # Cut-off frequency (-3 dB) [Hz]

print("=" * 55)
print("  First-Order RC Low-Pass Filter")
print("=" * 55)
print(f"  R        = {R:.1f} Ω  ({R/1e3:.2f} kΩ)")
print(f"  C        = {C:.2e} F  ({C*1e9:.1f} nF)")
print(f"  τ (RC)   = {tau:.4e} s  ({tau*1e6:.2f} µs)")
print(f"  f_c      = {fc:.2f} Hz")
print("=" * 55)

# ============================================================
# 1. Gain Magnitude & Phase Shift at 500 Hz
# ============================================================
f_target = 500.0  # Hz
w_target = 2.0 * np.pi * f_target
H = 1.0 / (1.0 + 1j * w_target * tau)   # H(jω) = 1 / (1 + jωRC)

mag = np.abs(H)
mag_dB = 20.0 * np.log10(mag)
phase_deg = np.angle(H, deg=True)

print(f"\n--- At f = {f_target:.0f} Hz ---")
print(f"  |H|      = {mag:.6f}  ({mag_dB:.4f} dB)")
print(f"  ∠H       = {phase_deg:.4f}°  ({phase_deg:.2f}°)")
print(f"  Attenuation = {abs(mag_dB):.2f} dB")

# ============================================================
# 2. Alias Frequency Calculation
#    ADC sampling rate fs = 2 kHz
#    Interfering sinusoid at 1.7 kHz
# ============================================================
fs = 2000.0         # Sampling frequency [Hz]
f_nyq = fs / 2.0    # Nyquist frequency [Hz]
f_useful = 100.0    # Useful signal [Hz]
f_interfere = 1700.0  # Interfering sinusoid [Hz]

# Alias formula: f_alias = |f - N*fs|, choose integer N to place in [0, fs/2]
# More precisely: f_alias = |f - round(f/fs) * fs|
N_alias = np.round(f_interfere / fs)
f_alias = np.abs(f_interfere - N_alias * fs)

print(f"\n--- Alias Calculation ---")
print(f"  ADC Sampling rate fs  = {fs/1e3:.0f} kHz")
print(f"  Nyquist frequency     = {f_nyq:.0f} Hz")
print(f"  Useful signal         = {f_useful:.0f} Hz")
print(f"  Interfering sinusoid  = {f_interfere:.0f} Hz")
print(f"  Folded by N           = {N_alias:.0f} × fs")
print(f"  Aliased frequency     = {f_alias:.0f} Hz")
print()

# --- Verify alias formula step by step ---
f_aliases = []
for k in [-1, 0, 1, 2]:
    fa = np.abs(f_interfere - k * fs)
    f_aliases.append((k, fa))
    in_band = "✓ in [0, fs/2]" if fa <= f_nyq else ""
    print(f"    k={k:+d}: |{f_interfere:.0f} - ({k:+d}×{fs:.0f})| = {fa:.0f} Hz  {in_band}")


  First-Order RC Low-Pass Filter
  R        = 2200.0 Ω  (2.20 kΩ)
  C        = 4.70e-08 F  (47.0 nF)
  τ (RC)   = 1.0340e-04 s  (103.40 µs)
  f_c      = 1539.22 Hz

--- At f = 500 Hz ---
  |H|      = 0.951079  (-0.4357 dB)
  ∠H       = -17.9959°  (-18.00°)
  Attenuation = 0.44 dB

--- Alias Calculation ---
  ADC Sampling rate fs  = 2 kHz
  Nyquist frequency     = 1000 Hz
  Useful signal         = 100 Hz
  Interfering sinusoid  = 1700 Hz
  Folded by N           = 1 × fs
  Aliased frequency     = 300 Hz

    k=-1: |1700 - (-1×2000)| = 3700 Hz  
    k=+0: |1700 - (+0×2000)| = 1700 Hz  
    k=+1: |1700 - (+1×2000)| = 300 Hz  ✓ in [0, fs/2]
    k=+2: |1700 - (+2×2000)| = 2300 Hz  
